In [ ]:
"""
针对每一个性状，分别为每一个 CB_Dynamic(year_location)
绘制 SHAP heatmap 图。

说明：
1. CB_FitAll 使用全部训练数据训练，但不绘制 CB_FitAll 的 SHAP heatmap。
2. CB_Dynamic 使用 CB_FitAll 作为 init_model，在每个 year/location 环境中继续训练，
   并基于该环境的训练数据计算 SHAP heatmap。
3. 模型仍然使用 location/year 作为环境变量。
4. heatmap 只展示每个 CB_Dynamic 模型 mean(abs(SHAP)) 排名前 15 的 fea_ 特征。
5. heatmap 横轴样本按模型输出值从低到高排序。
6. 不显示 “Sum of other features”。
"""

import os
import re
import numpy as np
import pandas as pd
import shap
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor, Pool


# =========================================================
# 基本参数
# =========================================================
IN_PATH = "/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/trainTestData/"

OUTDIR = "/data2/zhoujb/project/cropVarCefComb/processData/9-0-runOnline/shap_heatmap_all_models/"
os.makedirs(OUTDIR, exist_ok=True)

FONT_PATH = "/home/zhoujb/local/font/Times New Roman.ttf"

traits = [
    "Yield_per_plant",
    "1000-grain_weight",
    "Grain_length_to_width_ratio",
    "Heading_date",
    "Plant_height",
    "Grain_per_panicle",
    "Seed_set_rate",
    "Tiller_number"
]

cat_features = ["location", "year"]
min_env_samples = 5
TOP_N = 15

catboost_params = {
    "iterations": 1000,
    "thread_count": 48,
    "verbose": 0,
    "random_seed": 42
}


In [ ]:
# =========================================================
# 函数1：规范 year，去掉 .0
# =========================================================
def clean_year_value(y):
    return str(int(float(y)))


# =========================================================
# 函数2：文件名安全处理
# =========================================================
def safe_name(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))


# =========================================================
# 函数3：设置 Times New Roman 字体
# =========================================================
def set_times_new_roman_font(font_path):
    if os.path.exists(font_path):
        mpl.font_manager.fontManager.addfont(font_path)
        prop = mpl.font_manager.FontProperties(fname=font_path)
        font_name = prop.get_name()
    else:
        print(f"Warning: font file not found -> {font_path}")
        font_name = "Times New Roman"

    mpl.rcParams["font.family"] = font_name
    mpl.rcParams["font.sans-serif"] = ["Times New Roman"]
    mpl.rcParams["axes.unicode_minus"] = False
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42

    return font_name


# =========================================================
# 函数4：将 CatBoost SHAP 输出转换为 shap.Explanation
# 同时返回完整模型输出值，用于样本排序
# =========================================================
def get_catboost_shap_explanation(
    model,
    data_df,
    feature_cols,
    cat_features,
    keep_only_fea=True
):
    pool = Pool(
        data=data_df[feature_cols],
        cat_features=cat_features
    )

    shap_array = model.get_feature_importance(pool, type="ShapValues")
    shap_array = np.asarray(shap_array)

    # CatBoost SHAP 最后一列是 expected value
    shap_values = shap_array[:, :-1]
    base_values = shap_array[:, -1]

    # 完整模型输出值，用于横轴从低到高排序
    model_output = np.asarray(model.predict(pool))

    if keep_only_fea:
        keep_idx = [
            i for i, feature in enumerate(feature_cols)
            if feature.startswith("fea_")
        ]
    else:
        keep_idx = list(range(len(feature_cols)))

    kept_features = [feature_cols[i] for i in keep_idx]
    kept_feature_names = [
        feature.replace("fea_", "", 1) for feature in kept_features
    ]

    shap_value = shap.Explanation(
        values=shap_values[:, keep_idx],
        base_values=base_values,
        data=data_df[kept_features].values,
        feature_names=kept_feature_names
    )

    return shap_value, model_output


# =========================================================
# 函数5：只保留前 TOP_N 特征，并按模型输出从低到高排序样本
# =========================================================
def prepare_heatmap_shap_value(shap_value, model_output, top_n=15):
    values = np.asarray(shap_value.values)
    data = np.asarray(shap_value.data)
    model_output = np.asarray(model_output).reshape(-1)

    n_features = values.shape[1]
    top_n = min(top_n, n_features)

    # 每个模型自己的前 top_n 个重要特征
    mean_abs_shap = np.abs(values).mean(axis=0)
    top_idx = np.argsort(mean_abs_shap)[::-1][:top_n]

    # 样本按完整模型输出值从低到高排序
    sample_idx = np.argsort(model_output)

    selected_values = values[sample_idx][:, top_idx]
    selected_data = data[sample_idx][:, top_idx]
    selected_output = model_output[sample_idx]

    # 为了让上方小曲线对应完整模型输出，
    # 将未显示特征的总贡献合并进每个样本的 base_values。
    selected_base_values = selected_output - selected_values.sum(axis=1)

    feature_names = list(np.asarray(shap_value.feature_names, dtype=object)[top_idx])

    return shap.Explanation(
        values=selected_values,
        base_values=selected_base_values,
        data=selected_data,
        feature_names=feature_names
    )


# =========================================================
# 函数6：绘制并保存 SHAP heatmap
# =========================================================
def save_shap_heatmap(shap_value, fig_path, font_name):
    cmap = sns.color_palette("vlag", as_cmap=True)

    with plt.rc_context():
        mpl.rcParams["font.family"] = font_name
        mpl.rcParams["font.sans-serif"] = ["Times New Roman"]
        mpl.rcParams["axes.unicode_minus"] = False
        plt.rcParams["pdf.fonttype"] = 42
        plt.rcParams["ps.fonttype"] = 42

        # shap_value 已经提前按模型输出排序，所以这里固定原顺序，不再聚类重排
        shap.plots.heatmap(
            shap_value,
            max_display=shap_value.values.shape[1],
            instance_order=np.arange(shap_value.values.shape[0]),
            show=False,
            cmap=cmap
        )

        plt.savefig(
            fig_path,
            format="pdf",
            bbox_inches="tight",
            transparent=True
        )
        plt.close()


# =========================================================
# 主程序
# =========================================================
font_name = set_times_new_roman_font(FONT_PATH)

for target_col in traits:

    print(f"\n========== Running trait: {target_col} ==========")

    trait_outdir = os.path.join(OUTDIR, safe_name(target_col))
    os.makedirs(trait_outdir, exist_ok=True)

    train_file = os.path.join(IN_PATH, f"train-{target_col}.txt")
    if not os.path.exists(train_file):
        print(f"Skip {target_col}: file not found -> {train_file}")
        continue

    train_df = pd.read_table(train_file)

    train_df["location"] = train_df["location"].astype(str)
    train_df["year"] = train_df["year"].apply(clean_year_value)

    fea_cols = [c for c in train_df.columns if c.startswith("fea_")]
    model_feature_cols = fea_cols + cat_features

    # year × location 组合，环境变量仍然使用 location/year
    pairs = (
        train_df[["year", "location"]]
        .drop_duplicates()
        .sort_values(["year", "location"])
    )
    pairs = list(pairs.itertuples(index=False, name=None))

    # =====================================================
    # CB_FitAll：使用全部训练数据训练
    # 注意：这里只训练，不绘制 CB_FitAll 的 SHAP heatmap
    # =====================================================
    whole_pool = Pool(
        data=train_df[model_feature_cols],
        label=train_df[target_col],
        cat_features=cat_features
    )

    whole_model = CatBoostRegressor(**catboost_params)
    whole_model.fit(whole_pool)

    # =====================================================
    # CB_Dynamic：每个 year/location 环境分别训练并画图
    # =====================================================
    for year, location in pairs:

        year_clean = clean_year_value(year)

        train_data = train_df[
            (train_df["year"] == year_clean) &
            (train_df["location"] == location)
        ].copy()

        if len(train_data) < min_env_samples:
            print(f"Skip {year_clean}_{location}: too few samples ({len(train_data)})")
            continue

        train_pool = Pool(
            data=train_data[model_feature_cols],
            label=train_data[target_col],
            cat_features=cat_features
        )

        dynamic_model = CatBoostRegressor(**catboost_params)
        dynamic_model.fit(train_pool, init_model=whole_model)

        dynamic_shap_value, dynamic_output = get_catboost_shap_explanation(
            model=dynamic_model,
            data_df=train_data,
            feature_cols=model_feature_cols,
            cat_features=cat_features,
            keep_only_fea=True
        )

        heatmap_shap_value = prepare_heatmap_shap_value(
            shap_value=dynamic_shap_value,
            model_output=dynamic_output,
            top_n=TOP_N
        )

        dynamic_fig_path = os.path.join(
            trait_outdir,
            f"{safe_name(target_col)}_CB_Dynamic_{safe_name(year_clean)}_{safe_name(location)}_SHAP_heatmap_top{TOP_N}_sorted.pdf"
        )

        save_shap_heatmap(
            shap_value=heatmap_shap_value,
            fig_path=dynamic_fig_path,
            font_name=font_name
        )

        print(f"Saved CB_Dynamic heatmap: {dynamic_fig_path}")

print("\nAll done.")